In [2]:
import pandas as pd

URL = "https://berkeley-earth-temperature.s3.us-west-1.amazonaws.com/Regional/TAVG/france-TAVG-Trend.txt"

df = pd.read_fwf(
    URL,
    comment='%',
    header=None
)

df.columns = [
    "year", "month",
    "anomaly_monthly", "uncertainty_monthly",
    "anomaly_annual", "uncertainty_annual",
    "anomaly_5yr", "uncertainty_5yr",
    "anomaly_10yr", "uncertainty_10yr",
    "anomaly_20yr", "uncertainty_20yr"
]

df.to_csv("france_TAVG_trend.csv", index=False)

print(df.head())
print(df.shape)


   year month anomaly_monthly uncertainty_monthly anomaly_annual  \
0  This     e          ains a               extra          regio   
1  temp    ur           sults               oduce          the B   
2  meth    or           regio                 NaN            NaN   
3     F     e             NaN                 NaN            NaN   
4   The    el          arth m               hod t          tempe   

  uncertainty_annual anomaly_5yr uncertainty_5yr anomaly_10yr  \
0              l sum          la             urf          NaN   
1              keley         ave              ng          NaN   
2                NaN         NaN             NaN          NaN   
3                NaN         NaN             NaN          NaN   
4               ture         tio             rom            e   

  uncertainty_10yr anomaly_20yr uncertainty_20yr  
0              NaN          NaN              NaN  
1              NaN          NaN              NaN  
2              NaN          NaN              Na

In [3]:
# Nettoyage du CSV généré

INPUT_CSV = "france_TAVG_trend.csv"
OUTPUT_CSV = "france_TAVG_trend_clean.csv"

df = pd.read_csv(INPUT_CSV)

# Conversion forcée en numérique
df["year"] = pd.to_numeric(df["year"], errors="coerce")

# Filtrage logique
df_clean = df[df["year"] >= 1850].copy()

# Reset index
df_clean.reset_index(drop=True, inplace=True)

# Export
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"CSV nettoyé généré : {OUTPUT_CSV}")
print("Nombre de lignes :", len(df_clean))
print("Années couvertes :", df_clean.year.min(), "-", df_clean.year.max())


CSV nettoyé généré : france_TAVG_trend_clean.csv
Nombre de lignes : 2052
Années couvertes : 1850.0 - 2020.0


In [5]:
# Agrégation mensuelle en saisonnière et annuelle
INPUT_CSV = "france_TAVG_trend_clean.csv"
OUTPUT_CSV = "france_TAVG_trend_monthly_seasonal.csv"

# -------------------------------------------------------------------
# 1. Lecture et types
# -------------------------------------------------------------------
df = pd.read_csv(INPUT_CSV)

df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df["anomaly_monthly"] = pd.to_numeric(df["anomaly_monthly"], errors="coerce")

# -------------------------------------------------------------------
# 2. Pivot
# -------------------------------------------------------------------
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

df["month_name"] = df["month"].map(month_map)

pivot = (
    df.pivot(index="year", columns="month_name", values="anomaly_monthly")
      .reset_index()
)

# -------------------------------------------------------------------
# 3. Agrégations STRICTES (sans min_count)
# -------------------------------------------------------------------

months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# J-D
jd = pivot[months].mean(axis=1)
jd[pivot[months].count(axis=1) < 12] = pd.NA
pivot["J-D"] = jd

# Décembre année N-1
pivot["Dec_prev"] = pivot["Dec"].shift(1)

# D-N
dn_months = ["Dec_prev","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]
dn = pivot[dn_months].mean(axis=1)
dn[pivot[dn_months].count(axis=1) < 12] = pd.NA
pivot["D-N"] = dn

# DJF
djf_months = ["Dec_prev","Jan","Feb"]
djf = pivot[djf_months].mean(axis=1)
djf[pivot[djf_months].count(axis=1) < 3] = pd.NA
pivot["DJF"] = djf

# MAM
mam_months = ["Mar","Apr","May"]
mam = pivot[mam_months].mean(axis=1)
mam[pivot[mam_months].count(axis=1) < 3] = pd.NA
pivot["MAM"] = mam

# JJA
jja_months = ["Jun","Jul","Aug"]
jja = pivot[jja_months].mean(axis=1)
jja[pivot[jja_months].count(axis=1) < 3] = pd.NA
pivot["JJA"] = jja

# SON
son_months = ["Sep","Oct","Nov"]
son = pivot[son_months].mean(axis=1)
son[pivot[son_months].count(axis=1) < 3] = pd.NA
pivot["SON"] = son

# Nettoyage
pivot.drop(columns=["Dec_prev"], inplace=True)

# -------------------------------------------------------------------
# 4. Ordre final
# -------------------------------------------------------------------
final_columns = [
    "year","Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec",
    "J-D","D-N","DJF","MAM","JJA","SON"
]

pivot = pivot[final_columns]
pivot.rename(columns={"year": "Year"}, inplace=True)

# -------------------------------------------------------------------
# 5. Export
# -------------------------------------------------------------------
pivot.to_csv(OUTPUT_CSV, index=False)

print(f"CSV généré : {OUTPUT_CSV}")
print("Années :", pivot.Year.min(), "-", pivot.Year.max())


CSV généré : france_TAVG_trend_monthly_seasonal.csv
Années : 1850 - 2020
